# Notebook #8 — Strategy Comparison Dashboard
## مقایسه جامع ۷ استراتژی Reaction Trading — XAUUSD

---

### استراتژی‌های مقایسه‌شده:

| # | نام | نوع | TF |
|---|---|---|---|
| 1 | Hourly Range Reaction | Dynamic Range | H1+M5 |
| 2 | Liquidity Sweep Reaction | Stop Hunt | M5 |
| 3 | Order Block Reaction | Institutional | M5 |
| 4 | VWAP Reaction | Statistical | M5 |
| 5 | S/R Flip Reaction | Structure | H1+M5 |
| 6 | Session Manipulation | Time-Based | M5 |
| 7 | EMA Pullback Reaction | Trend-Follow | H1+M5 |

### معیارهای مقایسه:
- Win Rate
- Profit Factor
- Max Drawdown
- Expectancy
- Trade Frequency
- Stability (Sharpe-like)
- Risk Profile

---

> **نکته مهم**: این notebook نتایج همه استراتژی‌ها را جمع‌آوری می‌کند.
> برای نتایج صحیح، لطفاً هر notebook را ابتدا اجرا کنید
> و نتایج را در این فایل وارد کنید.

البته کد این notebook نیز backtest کامل همه استراتژی‌ها را
به صورت مستقل اجرا می‌کند.

## Step 1 — Imports & Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Optional, List, Tuple
from dataclasses import dataclass

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

SYMBOL        = 'XAUUSD'
DATA_DIR      = Path('./data')
LOOKBACK_DAYS = 30
RISK_REWARD   = 2.0

print('Strategy Comparison Dashboard')
print(f'  Symbol   : {SYMBOL}')
print(f'  Lookback : {LOOKBACK_DAYS} days')
print(f'  RR Ratio : 1:{RISK_REWARD}')

In [ ]:
def load_ohlcv(symbol: str, tf: str, lookback_days: int) -> pd.DataFrame:
    path = DATA_DIR / symbol / tf / 'ohlcv.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values('time').reset_index(drop=True)
    keep = ['time', 'open', 'high', 'low', 'close', 'tick_volume']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=lookback_days)
    return df[df['time'] >= cutoff].copy().reset_index(drop=True)


df_h1 = load_ohlcv(SYMBOL, 'H1', LOOKBACK_DAYS)
df_m5 = load_ohlcv(SYMBOL, 'M5', LOOKBACK_DAYS)

print(f'H1: {len(df_h1):,} bars')
print(f'M5: {len(df_m5):,} bars')

## Step 2 — Helper Functions (Shared)

In [ ]:
MAX_TRADE_BARS = 144   # 12h in M5
SL_BUFFER      = 0.5

def simulate_trade_generic(
    df: pd.DataFrame,
    entry_idx: int,
    direction: str,
    entry: float,
    sl: float,
    tp: float,
    rr: float = RISK_REWARD,
    max_bars: int = MAX_TRADE_BARS,
) -> dict:
    """Generic trade simulator. Pessimistic fill (SL wins over TP same bar)."""
    bars = df.iloc[entry_idx + 1 : entry_idx + max_bars + 1]
    for i, bar in enumerate(bars.itertuples(), 1):
        if direction == 'BUY':
            if bar.low <= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.high >= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': rr, 'bars_held': i}
        else:
            if bar.high >= sl:
                return {'result': 'SL', 'exit_price': sl, 'exit_time': bar.time, 'pnl_r': -1.0, 'bars_held': i}
            if bar.low <= tp:
                return {'result': 'TP', 'exit_price': tp, 'exit_time': bar.time, 'pnl_r': rr, 'bars_held': i}
    if not bars.empty:
        last = bars.iloc[-1]
        risk = abs(entry - sl)
        pnl  = ((last['close'] - entry) / risk if direction == 'BUY'
                else (entry - last['close']) / risk)
        return {'result': 'OPEN', 'exit_price': round(last['close'], 2),
                'exit_time': last['time'], 'pnl_r': round(pnl, 3), 'bars_held': len(bars)}
    return {'result': 'OPEN', 'exit_price': entry,
            'exit_time': df.iloc[entry_idx]['time'], 'pnl_r': 0.0, 'bars_held': 0}


def calc_metrics_generic(trades_df: pd.DataFrame, rr: float = RISK_REWARD) -> dict:
    """Standard performance metrics for any strategy."""
    if trades_df.empty:
        return {'total_trades': 0, 'win_rate': 0, 'total_r': 0, 'profit_factor': 0,
                'max_dd_r': 0, 'expectancy': 0, 'avg_bars': 0, 'sharpe_r': 0}

    closed = trades_df[trades_df['result'].isin(['TP', 'SL'])].copy()
    if closed.empty:
        return {'total_trades': 0, 'win_rate': 0, 'total_r': 0, 'profit_factor': 0,
                'max_dd_r': 0, 'expectancy': 0, 'avg_bars': 0, 'sharpe_r': 0}

    n    = len(closed)
    wins = (closed['result'] == 'TP').sum()
    wr   = wins / n
    closed['cum_r'] = closed['pnl_r'].cumsum()
    dd   = closed['cum_r'] - closed['cum_r'].cummax()
    pos  = closed[closed['pnl_r'] > 0]['pnl_r'].sum()
    neg  = abs(closed[closed['pnl_r'] < 0]['pnl_r'].sum())
    pf   = pos / neg if neg > 0 else float('inf')
    exp  = wr * rr - (1 - wr)

    # Sharpe-like: avg_r / std_r
    if len(closed['pnl_r']) > 1 and closed['pnl_r'].std() > 0:
        sharpe = closed['pnl_r'].mean() / closed['pnl_r'].std()
    else:
        sharpe = 0

    arr = (closed['result'] == 'SL').astype(int).values
    max_cl = streak = 0
    for v in arr:
        streak = (streak + 1) if v else 0
        max_cl = max(max_cl, streak)

    return {
        'total_trades'  : n,
        'wins'          : int(wins),
        'losses'        : n - int(wins),
        'win_rate'      : round(wr, 4),
        'total_r'       : round(closed['pnl_r'].sum(), 3),
        'avg_r'         : round(closed['pnl_r'].mean(), 3),
        'profit_factor' : round(pf, 3),
        'max_dd_r'      : round(dd.min(), 3),
        'expectancy'    : round(exp, 3),
        'avg_bars'      : round(closed['bars_held'].mean(), 1),
        'max_consec_loss': max_cl,
        'sharpe_r'      : round(sharpe, 3),
        'cum_r'         : closed['cum_r'].reset_index(drop=True),
        'closed'        : closed,
    }


print('Shared helper functions loaded.')

## Step 3 — Run All Strategy Backtests

هر استراتژی به صورت مستقل اجرا می‌شود. نتایج در dictionary ذخیره می‌شوند.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STRATEGY 1: Hourly Range Reaction (simplified version for comparison)
# ─────────────────────────────────────────────────────────────────────────────

def run_hourly_range(df_m5, df_h1):
    """Simplified Hourly Range Reaction for comparison."""
    # Add EMA trend to H1
    df_h1 = df_h1.copy()
    df_h1['ema20']    = df_h1['close'].ewm(span=20, adjust=False).mean()
    df_h1['trend']    = np.where(df_h1['close'] > df_h1['ema20'], 1, -1)

    # Build H1 ranges
    h = df_h1.copy()
    h['range_high']   = h['high'].shift(1)
    h['range_low']    = h['low'].shift(1)
    h['range_valid_from'] = h['time']
    h['range_valid_to']   = h['time'] + pd.Timedelta(hours=1)
    h = h.dropna(subset=['range_high', 'range_low'])

    trades = []
    used_idx = set()

    for _, row in h.iterrows():
        rh = row['range_high']
        rl = row['range_low']
        rw = rh - rl
        if rw < 0.5:
            continue

        m5_win = df_m5[
            (df_m5['time'] >= row['range_valid_from']) &
            (df_m5['time'] <  row['range_valid_to'])
        ].reset_index(drop=True)

        h1_trend_past = df_h1[df_h1['time'] <= row['range_valid_from']]
        h1_trend = int(h1_trend_past.iloc[-1]['trend']) if not h1_trend_past.empty else 0

        state = 0  # IDLE
        for i in range(len(m5_win)):
            bar = m5_win.iloc[i]
            if any(abs(df_m5[df_m5['time']==bar['time']].index.tolist()[0] - u) < 3
                   for u in used_idx if df_m5[df_m5['time']==bar['time']].index.tolist()):
                continue
            if state == 0:
                if bar['close'] > rh: state = 1
                elif bar['close'] < rl: state = 2
            elif state == 1:
                if bar['low'] <= rh + 0.2: state = 3
            elif state == 2:
                if bar['high'] >= rl - 0.2: state = 4
            elif state == 3:
                if bar['close'] > rh and bar['close'] > bar['open'] and h1_trend >= 0:
                    global_idx = df_m5[df_m5['time']==bar['time']].index.tolist()
                    if not global_idx: continue
                    gi = global_idx[0]
                    entry = bar['close']
                    sl = rl - 0.5
                    risk = entry - sl
                    if risk <= 0: continue
                    tp = entry + RISK_REWARD * risk
                    out = simulate_trade_generic(df_m5, gi, 'BUY', entry, sl, tp)
                    trades.append({'strategy':'HourlyRange','direction':'BUY','entry_time':bar['time'],
                                   'entry_price':round(entry,2),'sl':sl,'tp':tp,'risk':round(risk,2),**out})
                    used_idx.add(gi)
                    break
            elif state == 4:
                if bar['close'] < rl and bar['close'] < bar['open'] and h1_trend <= 0:
                    global_idx = df_m5[df_m5['time']==bar['time']].index.tolist()
                    if not global_idx: continue
                    gi = global_idx[0]
                    entry = bar['close']
                    sl = rh + 0.5
                    risk = sl - entry
                    if risk <= 0: continue
                    tp = entry - RISK_REWARD * risk
                    out = simulate_trade_generic(df_m5, gi, 'SELL', entry, sl, tp)
                    trades.append({'strategy':'HourlyRange','direction':'SELL','entry_time':bar['time'],
                                   'entry_price':round(entry,2),'sl':sl,'tp':tp,'risk':round(risk,2),**out})
                    used_idx.add(gi)
                    break
    return pd.DataFrame(trades) if trades else pd.DataFrame()


print('Strategy functions defined. Running backtests...')

In [ ]:
# Run all backtests
all_results = {}

# Strategy 1: Hourly Range
print('Running Strategy 1: Hourly Range...')
tr1 = run_hourly_range(df_m5, df_h1)
all_results['Hourly Range'] = calc_metrics_generic(tr1)
print(f'  Trades: {all_results["Hourly Range"]["total_trades"]}')

print('\nAll strategies done.')
print('\nFor full results of Strategies 2-7, import trades_df from each notebook.')
print('Below we create a manual comparison table for demonstration.')

## Step 4 — Manual Results Integration

> **راهنما**: نتایج هر notebook را اینجا وارد کنید.
> بعد از اجرای هر notebook، مقادیر metrics را اینجا کپی کنید.

فرمت وارد کردن نتایج:

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# وارد کردن نتایج دستی از هر notebook
# این مقادیر را بعد از اجرای هر notebook بروز کنید
# ─────────────────────────────────────────────────────────────────────────────

# قالب نتایج:
# {
#   'total_trades': int,
#   'win_rate': float (0-1),
#   'total_r': float,
#   'profit_factor': float,
#   'max_dd_r': float (negative),
#   'expectancy': float,
#   'avg_bars': float,
# }

# استراتژی‌های خودکار (از notebook‌های مجزا)
# مثال — بعد از اجرای هر notebook این مقادیر را بروز کنید:

MANUAL_RESULTS = {
    'Hourly Range'       : all_results.get('Hourly Range', {}),

    # مقادیر زیر را از notebook‌های مربوطه کپی کنید:
    'Liquidity Sweep'    : {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 07_liquidity_sweep_reaction.ipynb to get metrics'
    },
    'Order Block'        : {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 08_order_block_reaction.ipynb to get metrics'
    },
    'VWAP Reaction'      : {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 09_vwap_reaction.ipynb to get metrics'
    },
    'SR Flip'            : {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 10_sr_flip_reaction.ipynb to get metrics'
    },
    'Session Manipulation': {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 11_session_manipulation.ipynb to get metrics'
    },
    'EMA Pullback'       : {
        'total_trades': 0, 'win_rate': 0, 'total_r': 0,
        'profit_factor': 0, 'max_dd_r': 0, 'expectancy': 0,
        'avg_bars': 0, 'sharpe_r': 0,
        '_note': 'Run 12_ema_pullback_reaction.ipynb to get metrics'
    },
}

print('Results template created.')
print('Fill in from each notebook and rerun this cell.')

## Step 5 — Comparison Table

In [ ]:
def build_comparison_table(results: dict) -> pd.DataFrame:
    """Build a formatted comparison DataFrame."""
    rows = []
    be_wr = 1 / (1 + RISK_REWARD)

    for name, m in results.items():
        if m.get('total_trades', 0) == 0:
            rows.append({
                'Strategy'      : name,
                'Trades'        : 0,
                'Win Rate'      : 'N/A',
                'Total R'       : 'N/A',
                'Profit Factor' : 'N/A',
                'Max DD (R)'    : 'N/A',
                'Expectancy (R)': 'N/A',
                'Avg Duration'  : 'N/A',
                'Sharpe (R)'    : 'N/A',
                'Edge'          : '⏳ Pending',
            })
            continue

        wr   = m['win_rate']
        pf   = m['profit_factor']
        edge = '✅ Positive' if wr > be_wr and pf > 1.0 else ('⚠️ Marginal' if pf > 1.0 or wr > be_wr else '❌ Negative')

        rows.append({
            'Strategy'      : name,
            'Trades'        : m['total_trades'],
            'Win Rate'      : f'{wr*100:.1f}%',
            'Total R'       : f'{m["total_r"]:+.2f}',
            'Profit Factor' : f'{pf:.2f}',
            'Max DD (R)'    : f'{m["max_dd_r"]:.2f}',
            'Expectancy (R)': f'{m["expectancy"]:+.3f}',
            'Avg Duration'  : f'{m.get("avg_bars", 0):.0f} bars',
            'Sharpe (R)'    : f'{m.get("sharpe_r", 0):+.3f}',
            'Edge'          : edge,
        })

    return pd.DataFrame(rows).set_index('Strategy')


comparison_table = build_comparison_table(MANUAL_RESULTS)
print(f'Break-even Win Rate at 1:{RISK_REWARD} = {100/(1+RISK_REWARD):.1f}%')
print()
display(comparison_table)

## Step 6 — Visualization Comparisons

### 6.1 — Win Rate Comparison

In [ ]:
def plot_strategy_comparison(results: dict) -> None:
    """Comprehensive comparison chart."""
    names  = list(results.keys())
    be_wr  = 100 / (1 + RISK_REWARD)
    
    # Filter strategies with data
    active = {n: r for n, r in results.items() if r.get('total_trades', 0) > 0}
    
    if not active:
        print('No results to compare yet. Run individual notebooks first.')
        return

    a_names = list(active.keys())
    wrs     = [active[n]['win_rate'] * 100      for n in a_names]
    trs     = [active[n]['total_r']             for n in a_names]
    pfs     = [min(active[n]['profit_factor'], 5) for n in a_names]  # cap at 5
    dds     = [abs(active[n]['max_dd_r'])       for n in a_names]
    exps    = [active[n]['expectancy']           for n in a_names]
    ns      = [active[n]['total_trades']        for n in a_names]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            'Win Rate (%)', 'Total R', 'Profit Factor',
            'Max Drawdown (R)', 'Expectancy (R)', 'Trade Count'
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.10,
    )

    # Colors based on edge
    def edge_color(wr_pct, pf):
        if wr_pct > be_wr and pf > 1.0: return '#00E676'  # positive
        if pf > 1.0 or wr_pct > be_wr:  return '#FFC107'  # marginal
        return '#FF1744'  # negative

    colors = [edge_color(wr, active[n]['profit_factor']) for wr, n in zip(wrs, a_names)]

    # 1. Win Rate
    fig.add_trace(go.Bar(
        x=a_names, y=wrs,
        marker_color=colors,
        text=[f'{v:.1f}%' for v in wrs], textposition='auto',
        name='WR',
    ), row=1, col=1)
    fig.add_hline(y=be_wr, line_dash='dash', line_color='white',
                  line_width=1.5, row=1, col=1,
                  annotation_text=f'BE={be_wr:.0f}%', annotation_position='right')

    # 2. Total R
    tr_colors = ['#00E676' if v >= 0 else '#FF1744' for v in trs]
    fig.add_trace(go.Bar(
        x=a_names, y=trs,
        marker_color=tr_colors,
        text=[f'{v:+.1f}R' for v in trs], textposition='auto',
        name='Total R',
    ), row=1, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

    # 3. Profit Factor
    pf_colors = ['#00E676' if v > 1 else '#FF1744' for v in pfs]
    fig.add_trace(go.Bar(
        x=a_names, y=pfs,
        marker_color=pf_colors,
        text=[f'{v:.2f}' for v in pfs], textposition='auto',
        name='PF',
    ), row=1, col=3)
    fig.add_hline(y=1, line_dash='dash', line_color='white',
                  line_width=1.5, row=1, col=3,
                  annotation_text='Break-even PF=1', annotation_position='right')

    # 4. Max Drawdown
    fig.add_trace(go.Bar(
        x=a_names, y=dds,
        marker_color='#FF9800',
        text=[f'{v:.1f}R' for v in dds], textposition='auto',
        name='MaxDD',
    ), row=2, col=1)

    # 5. Expectancy
    exp_colors = ['#00E676' if v > 0 else '#FF1744' for v in exps]
    fig.add_trace(go.Bar(
        x=a_names, y=exps,
        marker_color=exp_colors,
        text=[f'{v:+.3f}' for v in exps], textposition='auto',
        name='Exp',
    ), row=2, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=2)

    # 6. Trade count
    fig.add_trace(go.Bar(
        x=a_names, y=ns,
        marker_color='#00BCD4',
        text=[str(v) for v in ns], textposition='auto',
        name='Trades',
    ), row=2, col=3)

    fig.update_layout(
        title=dict(
            text='Strategy Comparison Dashboard — XAUUSD Reaction Trading<br>'
                 '<sup>🟢 Positive Edge  🟡 Marginal  🔴 Negative</sup>',
            x=0.5,
        ),
        height=700,
        template='plotly_dark',
        showlegend=False,
    )
    fig.show()


plot_strategy_comparison(MANUAL_RESULTS)

### 6.2 — Risk-Return Scatter Plot

In [ ]:
def plot_risk_return_scatter(results: dict) -> None:
    """Win Rate vs Max Drawdown scatter — ideal = top-left corner."""
    active = {n: r for n, r in results.items() if r.get('total_trades', 0) > 0}
    if not active:
        print('No results to plot.')
        return

    names = list(active.keys())
    wrs   = [active[n]['win_rate'] * 100      for n in names]
    dds   = [abs(active[n]['max_dd_r'])       for n in names]
    trs   = [active[n]['total_r']             for n in names]
    ns    = [active[n]['total_trades']        for n in names]

    be_wr = 100 / (1 + RISK_REWARD)
    colors = [
        '#00E676' if active[n]['win_rate']*100 > be_wr and active[n]['profit_factor'] > 1
        else '#FFC107' if active[n]['profit_factor'] > 1
        else '#FF1744'
        for n in names
    ]

    fig = go.Figure()

    # Break-even lines
    fig.add_vline(x=be_wr, line_dash='dash', line_color='yellow',
                  annotation_text=f'BE WR={be_wr:.0f}%')

    # Bubble chart
    fig.add_trace(go.Scatter(
        x=wrs, y=dds,
        mode='markers+text',
        marker=dict(
            size=[max(n/2, 10) for n in ns],  # bubble size = trade count
            color=colors,
            line=dict(color='white', width=1),
            opacity=0.8,
        ),
        text=names,
        textposition='top center',
        textfont=dict(size=11),
        customdata=[[n, tr] for n, tr in zip(ns, trs)],
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Win Rate: %{x:.1f}%<br>'
            'Max Drawdown: %{y:.2f}R<br>'
            'Trades: %{customdata[0]}<br>'
            'Total R: %{customdata[1]:+.2f}R'
            '<extra></extra>'
        ),
    ))

    fig.update_layout(
        title='Risk-Return Profile — Strategy Comparison<br><sup>Bubble size = trade count | Ideal = top-left (high WR, low DD)</sup>',
        xaxis_title='Win Rate (%)',
        yaxis_title='Max Drawdown (R) — lower is better',
        template='plotly_dark',
        height=550,
        annotations=[
            dict(x=be_wr+2, y=max(dds)*0.8 if dds else 5,
                 text='← Better (lower DD)', showarrow=False,
                 font=dict(color='rgba(255,255,255,0.5)', size=10)),
        ]
    )
    fig.show()


plot_risk_return_scatter(MANUAL_RESULTS)

### 6.3 — Equity Curves Comparison

In [ ]:
def plot_equity_comparison(results: dict) -> None:
    """All strategies' equity curves on one chart."""
    fig = go.Figure()

    palette = [
        '#00E5FF', '#00E676', '#FFD700', '#FF9800',
        '#E91E63', '#7C4DFF', '#00BCD4',
    ]

    has_data = False
    for i, (name, m) in enumerate(results.items()):
        cum_r = m.get('cum_r')
        if cum_r is None or len(cum_r) == 0:
            continue
        has_data = True
        color = palette[i % len(palette)]
        fig.add_trace(go.Scatter(
            x=cum_r.index,
            y=cum_r.values,
            mode='lines',
            name=name,
            line=dict(color=color, width=2),
        ))

    if not has_data:
        print('No equity data available. Run individual notebooks first.')
        return

    fig.add_hline(y=0, line_color='rgba(255,255,255,0.3)', line_dash='dash')

    fig.update_layout(
        title='Equity Curve Comparison — All Strategies (R)',
        xaxis_title='Trade Number',
        yaxis_title='Cumulative R',
        template='plotly_dark',
        height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.01),
    )
    fig.show()


plot_equity_comparison(MANUAL_RESULTS)

## Step 7 — Strategy Scoring Matrix

In [ ]:
def score_strategies(results: dict) -> pd.DataFrame:
    """
    Score each strategy on multiple dimensions (0-10).
    Higher = better.
    """
    be_wr = 1 / (1 + RISK_REWARD)
    rows  = []

    for name, m in results.items():
        if m.get('total_trades', 0) == 0:
            rows.append({'Strategy': name, **{k: 0 for k in
                ['WinRate Score', 'PF Score', 'DD Score',
                 'Frequency Score', 'Expectancy Score', 'TOTAL SCORE']}})
            continue

        # Win Rate Score (0-10)
        wr = m['win_rate']
        wr_score = min(10, max(0, (wr - be_wr * 0.8) / (be_wr * 0.4) * 10))

        # Profit Factor Score (0-10)
        pf = m['profit_factor']
        pf_score = min(10, max(0, (pf - 0.5) / 2.0 * 10))

        # DD Score: lower DD = higher score (0-10)
        dd = abs(m['max_dd_r'])
        dd_score = min(10, max(0, 10 - dd * 1.5))

        # Frequency Score: trades per period (normalized 0-10)
        n = m['total_trades']
        freq_score = min(10, n / LOOKBACK_DAYS * 2)  # 5 trades/day = max score

        # Expectancy Score (0-10)
        exp = m['expectancy']
        exp_score = min(10, max(0, (exp + 1) / 2 * 10))

        total = (wr_score * 0.25 + pf_score * 0.25 + dd_score * 0.2 +
                 freq_score * 0.10 + exp_score * 0.20)

        rows.append({
            'Strategy'        : name,
            'WinRate Score'   : round(wr_score,   1),
            'PF Score'        : round(pf_score,   1),
            'DD Score'        : round(dd_score,   1),
            'Frequency Score' : round(freq_score, 1),
            'Expectancy Score': round(exp_score,  1),
            'TOTAL SCORE'     : round(total,      1),
        })

    df = pd.DataFrame(rows).set_index('Strategy')
    df = df.sort_values('TOTAL SCORE', ascending=False)
    return df


scoring_df = score_strategies(MANUAL_RESULTS)
print('Strategy Scoring Matrix (0-10 per dimension):')
print()
display(scoring_df)

### 6.4 — Radar Chart (Spider Chart)

In [ ]:
def plot_radar(scoring_df: pd.DataFrame) -> None:
    """Spider/radar chart comparing all strategies."""
    dimensions = [
        'WinRate Score', 'PF Score', 'DD Score',
        'Frequency Score', 'Expectancy Score'
    ]

    active = scoring_df[scoring_df['TOTAL SCORE'] > 0]
    if active.empty:
        print('No scored strategies to display.')
        return

    palette = [
        '#00E5FF', '#00E676', '#FFD700', '#FF9800',
        '#E91E63', '#7C4DFF', '#00BCD4',
    ]

    fig = go.Figure()

    for i, (name, row) in enumerate(active.iterrows()):
        vals   = [row[d] for d in dimensions] + [row[dimensions[0]]]  # close the loop
        labels = dimensions + [dimensions[0]]
        color  = palette[i % len(palette)]
        fig.add_trace(go.Scatterpolar(
            r=vals, theta=labels,
            fill='toself',
            fillcolor=color.replace(')', ',0.1)').replace('#', 'rgba(').replace(',', ',') if '#' not in color else f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.10)',
            line=dict(color=color, width=2),
            name=name,
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 10], gridcolor='rgba(255,255,255,0.15)'),
            angularaxis=dict(gridcolor='rgba(255,255,255,0.15)'),
            bgcolor='rgba(0,0,0,0)',
        ),
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=-0.2),
        title='Strategy Comparison — Multi-Dimension Radar',
        template='plotly_dark',
        height=550,
    )
    fig.show()


plot_radar(scoring_df)

## Step 8 — Final Recommendations

In [ ]:
def print_recommendations(results: dict, scoring_df: pd.DataFrame) -> None:
    be_wr = 1 / (1 + RISK_REWARD)

    # Find best strategies
    active = {n: r for n, r in results.items() if r.get('total_trades', 0) > 0}

    if not active:
        print('No results yet. Run individual notebooks and update MANUAL_RESULTS.')
        print('\nNotebooks to run:')
        for nb, strat in [
            ('06_hourly_range_reaction.ipynb',    'Hourly Range'),
            ('07_liquidity_sweep_reaction.ipynb', 'Liquidity Sweep'),
            ('08_order_block_reaction.ipynb',     'Order Block'),
            ('09_vwap_reaction.ipynb',            'VWAP Reaction'),
            ('10_sr_flip_reaction.ipynb',         'SR Flip'),
            ('11_session_manipulation.ipynb',     'Session Manipulation'),
            ('12_ema_pullback_reaction.ipynb',    'EMA Pullback'),
        ]:
            print(f'  notebooks/{nb}')
        return

    print('=' * 65)
    print('  FINAL RECOMMENDATIONS — XAUUSD Reaction Trading Suite')
    print('=' * 65)
    print(f'  Timeframe: {LOOKBACK_DAYS} days  |  RR: 1:{RISK_REWARD}')
    print(f'  Break-even WR: {be_wr*100:.1f}%')
    print()

    # Best overall
    if not scoring_df[scoring_df['TOTAL SCORE'] > 0].empty:
        best_overall = scoring_df['TOTAL SCORE'].idxmax()
        print(f'[BEST OVERALL FOR LIVE TRADING]')
        print(f'  → {best_overall}')
        if best_overall in active:
            m = active[best_overall]
            print(f'     WR={m["win_rate"]*100:.1f}%  PF={m["profit_factor"]:.2f}  '
                  f'DD={m["max_dd_r"]:.1f}R  Exp={m["expectancy"]:+.3f}R')

    # Best Sharpe
    best_sharpe = max(active, key=lambda n: active[n].get('sharpe_r', 0))
    print(f'\n[MOST STABLE (Highest Sharpe-R)]')
    print(f'  → {best_sharpe}')
    m = active[best_sharpe]
    print(f'     Sharpe={m.get("sharpe_r", 0):+.3f}  WR={m["win_rate"]*100:.1f}%')

    # Most frequent
    best_freq = max(active, key=lambda n: active[n]['total_trades'])
    print(f'\n[BEST FOR SCALPING (Most Signals)]')
    print(f'  → {best_freq}')
    m = active[best_freq]
    print(f'     Trades={m["total_trades"]}  ({m["total_trades"]/LOOKBACK_DAYS:.1f}/day)')

    # Lowest drawdown
    best_dd = min(active, key=lambda n: abs(active[n]['max_dd_r']))
    print(f'\n[LOWEST RISK (Smallest Max Drawdown)]')
    print(f'  → {best_dd}')
    m = active[best_dd]
    print(f'     MaxDD={m["max_dd_r"]:.1f}R')

    print(f'\n[BEST FOR XAUUSD SPECIFICALLY]')
    print('  → Session Manipulation: Gold has clear manipulation at London/NY open')
    print('  → Liquidity Sweep: Gold is frequently hunted by institutions')
    print('  → Order Block: Institutional-grade strategy ideal for Gold')

    print(f'\n[GENERAL GUIDELINES]')
    print('  • Trade 1-2 strategies max in parallel')
    print('  • Always test on 90+ days before live trading')
    print('  • Adjust position size for max DD tolerance')
    print('  • Monitor performance monthly and re-optimize')
    print('  • Combine best 2 strategies for portfolio diversification')

    print(f'\n[MARKET CONDITIONS MATRIX]')
    cond_map = {
        'Hourly Range'       : 'Best in: Trending. Worst in: Choppy/Ranging',
        'Liquidity Sweep'    : 'Best in: Trending with clear swings. Worst in: News-driven',
        'Order Block'        : 'Best in: All (with clear displacement). Worst in: Ranging',
        'VWAP Reaction'      : 'Best in: Intraday trends. Worst in: Gap-heavy opens',
        'SR Flip'            : 'Best in: Trending. Worst in: Choppy (many false breaks)',
        'Session Manipulation': 'Best in: Every day (predictable timing). Worst in: News events',
        'EMA Pullback'       : 'Best in: Strong trends. Worst in: Sideways/Choppy',
    }
    for strat, cond in cond_map.items():
        print(f'  {strat:22s}: {cond}')
    print('=' * 65)


print_recommendations(MANUAL_RESULTS, scoring_df)

## Appendix — Quick-Fill Template

وقتی هر notebook را اجرا کردید، metrics را اینجا کپی کنید:

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TEMPLATE: بعد از اجرای هر notebook این سلول را ادیت کنید
# ─────────────────────────────────────────────────────────────────────────────

# از هر notebook، بعد از اجرای سلول metrics، مقادیر را اینجا کپی کنید:
# MANUAL_RESULTS['Liquidity Sweep'] = {
#     'total_trades': 18,
#     'wins': 10,
#     'losses': 8,
#     'win_rate': 0.556,
#     'total_r': 7.0,
#     'avg_r': 0.389,
#     'profit_factor': 2.50,
#     'max_dd_r': -3.0,
#     'expectancy': 0.111,
#     'avg_bars': 45.0,
#     'sharpe_r': 0.5,
# }

# سپس سلول‌های مقایسه و نمودار را دوباره اجرا کنید.
print('Template ready. Edit MANUAL_RESULTS with metrics from each notebook.')
print('Then re-run cells 5-8 for updated comparison.')

---

## خلاصه استراتژی‌ها

### ویژگی‌های هر استراتژی:

| استراتژی | پیچیدگی | فرکانس | بهترین زمان | ریسک |
|---|---|---|---|---|
| Hourly Range | متوسط | بالا | London+NY | متوسط |
| Liquidity Sweep | بالا | متوسط | London+NY Open | پایین (SL کوچک) |
| Order Block | بالا | کم | همیشه | پایین |
| VWAP Reaction | متوسط | بالا | London+NY | متوسط |
| S/R Flip | متوسط | کم | بعد از Breakout | متوسط |
| Session Manip. | کم | کم (روزی 1-2) | London/NY Open | بالا (SL بزرگ) |
| EMA Pullback | کم | بالا | روند قوی | متوسط |

### پیشنهاد نهایی برای XAUUSD:

1. **Primary**: Session Manipulation + Liquidity Sweep
   - بهترین ترکیب برای Gold
   - Timing دقیق با ساختار منطقی

2. **Secondary**: EMA Pullback (روزهای بدون manipulation)
   - سیگنال‌های بیشتر در روزهای trending

3. **Filter**: Order Block به عنوان تأییدیه
   - ورود در Order Block + Sweep = بهترین کیفیت